# OpenPlaque experimental outer-wall / PAV prototype

Research prototype only. The candidate outer wall is **not clinically validated**. Review the overlays before interpreting any PAV value.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_DIR = Path('/content/OpenPlaque')
if REPO_DIR.exists():
    subprocess.run(['rm','-rf',str(REPO_DIR)], check=True)
subprocess.run(['git','clone','--branch','pav-outer-wall-prototype',
                'https://github.com/pazzani/OpenPlaque.git', str(REPO_DIR)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR/'src'))

import openplaque
from openplaque.pav import estimate_pav_from_labels, show_pav_overlay
print('OpenPlaque import OK:', openplaque.__file__)
print('PAV prototype import OK.')

## Load the UCLA study and artery masks

This version uses the same `Full_DICOM.zip` and artery-series mapping as the plaque-type notebook. It does **not** require a separate `CCTA_PATH` NIfTI file.


In [ ]:
import os
import numpy as np
import pandas as pd
import pydicom
import SimpleITK as sitk

from openplaque.study import OpenPlaqueStudy
from openplaque.run_new_data import auto_detect_or_fallback_series

DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
STUDY_ZIP = DRIVE_ROOT / 'Full_DICOM.zip'
MASK_DIR_CANDIDATES = [
    DRIVE_ROOT / 'UCLA_Plaque_Type_Estimates/nnunet_masks',
    DRIVE_ROOT / 'UCLA_Plaque_Context_Verification/nnunet_masks',
]
MASK_DIR = next((p for p in MASK_DIR_CANDIDATES
                 if all((p / f'{a}.nii.gz').exists() for a in ['LAD','LCX','RCA'])), None)

if not STUDY_ZIP.exists():
    raise FileNotFoundError(f'Missing {STUDY_ZIP}')
if MASK_DIR is None:
    raise FileNotFoundError('Could not find LAD.nii.gz, LCX.nii.gz, and RCA.nii.gz in expected folders.')

FALLBACK_SERIES = {'RCA': 1035, 'LCX': 1039, 'LAD': 1043}
VESSELS = ['LAD','LCX','RCA']
REFERENCE_SPACING_SERIES_NUMBER = 7
MANUAL_REFERENCE_SPACING_XYZ_MM = (0.3515625, 0.3515625, 0.3)
MAX_EXPECTED_CCTA_VOXEL_VOLUME_MM3 = 0.5

print('Study zip:', STUDY_ZIP)
print('Mask directory:', MASK_DIR)

In [ ]:
def spacing_from_dicom_files(files, fallback_spacing):
    if not files:
        return tuple(float(x) for x in fallback_spacing), 'sitk_fallback_no_files'
    datasets = []
    for path in files:
        try:
            datasets.append(pydicom.dcmread(path, stop_before_pixels=True, force=True))
        except Exception:
            pass
    if not datasets:
        return tuple(float(x) for x in fallback_spacing), 'sitk_fallback_no_readable_dicom'

    first = datasets[0]
    try:
        row_spacing, col_spacing = [float(x) for x in first.PixelSpacing]
    except Exception:
        row_spacing, col_spacing = float(fallback_spacing[1]), float(fallback_spacing[0])

    positions = []
    for ds in datasets:
        ipp = getattr(ds, 'ImagePositionPatient', None)
        if ipp is not None and len(ipp) == 3:
            positions.append(np.asarray([float(x) for x in ipp], dtype=float))
    z_spacing = None
    if len(positions) > 1:
        positions = sorted(positions, key=lambda p: tuple(p.tolist()))
        deltas = [float(np.linalg.norm(positions[i+1]-positions[i])) for i in range(len(positions)-1)]
        deltas = [d for d in deltas if d > 1e-6]
        if deltas:
            z_spacing = float(np.median(deltas))
    if z_spacing is None:
        for attr in ('SpacingBetweenSlices','SliceThickness'):
            v = getattr(first, attr, None)
            if v is not None:
                try:
                    z_spacing = float(v)
                    break
                except Exception:
                    pass
    if z_spacing is None:
        z_spacing = float(fallback_spacing[2])
    return (float(col_spacing), float(row_spacing), float(z_spacing)), 'dicom_metadata'

def dicom_files_for_series_number(study, series_number):
    matches = [s for s in study.series if int(s.get('series_number', -1)) == int(series_number)]
    if not matches:
        return []
    folder = matches[0]['folder']
    out = []
    for root, _, names in os.walk(folder):
        out.extend(os.path.join(root, n) for n in names)
    return out

study = OpenPlaqueStudy(str(STUDY_ZIP))
series_map = auto_detect_or_fallback_series(study, fallback_series=FALLBACK_SERIES)
series_map = {k:int(series_map[k]) for k in VESSELS}
print('Series map:', series_map)

ref_files = dicom_files_for_series_number(study, REFERENCE_SPACING_SERIES_NUMBER)
REFERENCE_SPACING, ref_source = spacing_from_dicom_files(ref_files, MANUAL_REFERENCE_SPACING_XYZ_MM)
if float(np.prod(REFERENCE_SPACING)) >= MAX_EXPECTED_CCTA_VOXEL_VOLUME_MM3:
    REFERENCE_SPACING = MANUAL_REFERENCE_SPACING_XYZ_MM
    ref_source = 'manual_reference_spacing'
print('Reference spacing:', REFERENCE_SPACING, ref_source)

In [ ]:
volumes = {}
masks = {}
spacings = {}
images = {}

for artery in VESSELS:
    image, volume, dicom_files = study.load_series(series_map[artery])
    mask_img = sitk.ReadImage(str(MASK_DIR / f'{artery}.nii.gz'))
    mask = sitk.GetArrayFromImage(mask_img)

    dicom_spacing, spacing_source = spacing_from_dicom_files(dicom_files, image.GetSpacing())
    analysis_spacing = dicom_spacing
    if float(np.prod(dicom_spacing)) >= MAX_EXPECTED_CCTA_VOXEL_VOLUME_MM3:
        analysis_spacing = REFERENCE_SPACING
        spacing_source = f'reference_series_{REFERENCE_SPACING_SERIES_NUMBER}'

    if volume.shape != mask.shape:
        raise ValueError(
            f'{artery}: DICOM artery volume shape {volume.shape} does not match mask shape {mask.shape}. '
            'The saved mask and detected DICOM series are not aligned.'
        )

    volumes[artery] = volume
    masks[artery] = mask
    spacings[artery] = analysis_spacing
    images[artery] = image

    print(f'{artery}: series={series_map[artery]}, shape={volume.shape}, '
          f'image spacing={image.GetSpacing()}, analysis spacing={analysis_spacing} ({spacing_source}), '
          f'labels={np.unique(mask)}')

## Run experimental PAV estimate

Start with a 2.0 mm maximum expansion and -30 HU fat threshold. These are exposed research parameters, not validated clinical defaults.


In [ ]:
MAX_WALL_THICKNESS_MM = 2.0
FAT_THRESHOLD_HU = -30.0

results = {}
for artery in VESSELS:
    r = estimate_pav_from_labels(
        volume=volumes[artery],
        mask=masks[artery],
        spacing=spacings[artery],
        max_wall_thickness_mm=MAX_WALL_THICKNESS_MM,
        fat_threshold_hu=FAT_THRESHOLD_HU,
    )
    results[artery] = r
    print('\n' + artery)
    r.summary()

## Visual review: highest-plaque slice for each artery


In [ ]:
import matplotlib.pyplot as plt
for artery in VESSELS:
    print(artery)
    show_pav_overlay(volumes[artery], masks[artery], results[artery].outer_wall_mask)
    plt.show()

## Visual review: three plaque-containing slices per artery


In [ ]:
for artery in VESSELS:
    mask = masks[artery]
    counts = np.sum(mask == 2, axis=(1,2))
    zs = np.where(counts > 0)[0]
    if len(zs) == 0:
        print(artery, ': no plaque-labeled slices')
        continue
    picks = np.unique(np.linspace(0, len(zs)-1, min(3, len(zs)), dtype=int))
    for idx in picks:
        z = int(zs[idx])
        print(f'{artery} z={z}, plaque voxels={counts[z]}')
        show_pav_overlay(volumes[artery], mask, results[artery].outer_wall_mask, z=z)
        plt.show()

## Summary table and whole-heart experimental PAV


In [ ]:
rows = []
for artery, r in results.items():
    rows.append({
        'artery': artery,
        'plaque_volume_mm3': r.plaque_volume_mm3,
        'outer_vessel_volume_mm3': r.outer_vessel_volume_mm3,
        'experimental_pav_percent': r.pav_percent,
        'analysis_spacing_xyz_mm': spacings[artery],
    })
df = pd.DataFrame(rows)
display(df)

total_plaque = df.plaque_volume_mm3.sum()
total_outer = df.outer_vessel_volume_mm3.sum()
whole_pav = 100 * total_plaque / total_outer if total_outer else 0
print(f'Total plaque volume: {total_plaque:.2f} mm^3')
print(f'Total candidate outer-vessel volume: {total_outer:.2f} mm^3')
print(f'Whole-heart experimental PAV: {whole_pav:.2f}%')
print('Do not interpret clinically until the outer-wall contours are validated.')

## Save candidate masks and CSV to Drive


In [ ]:
OUT_DIR = DRIVE_ROOT / 'PAV_Outer_Wall_Prototype'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for artery, r in results.items():
    out_img = sitk.GetImageFromArray(r.outer_wall_mask.astype(np.uint8))
    out_img.CopyInformation(images[artery])
    sitk.WriteImage(out_img, str(OUT_DIR / f'{artery}_candidate_outer_wall.nii.gz'))

df.to_csv(OUT_DIR / 'experimental_pav_by_artery.csv', index=False)
print('Saved results to:', OUT_DIR)